# First model: prepare the combined data

`combining_data.ipynb` already joins TMDB and IMDb, removes missing join IDs, and removes duplicate movie–director pairs. Start from its saved output instead of repeating the merge.

The combined file has multiple rows for co-directed movies. Group those directors before movie-level analysis. Only matched movies are available here; this does not restore the unmatched TMDB movies.


In [ ]:
import pandas as pd

combined = pd.read_csv(
    "combined_dataset.csv",
    dtype={"imdb_id": "string", "nconst": "string"},
)
print(f"Loaded {len(combined):,} movie–director rows")


## Keep one row per movie

Collect all IMDb directors for each movie, then attach that information to one copy of its movie characteristics.


In [ ]:
def join_names(values):
    return "|".join(values.dropna().drop_duplicates())

directors = combined.groupby("tmdb_id", as_index=False).agg(
    imdb_director_ids=("nconst", join_names),
    imdb_directors=("director_imdb", join_names),
    imdb_director_count=("nconst", "nunique"),
)

movies = (
    combined.drop(columns=["nconst", "director_imdb"])
    .drop_duplicates("tmdb_id")
    .merge(directors, on="tmdb_id", validate="one_to_one", sort=False)
)
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
assert movies["tmdb_id"].is_unique
print(f"Ready for analysis: {len(movies):,} unique movies")
display(movies.head())


## Check what still needs cleaning

Combining did not resolve missing financial information. Zero budget makes revenue/budget undefined; zero revenue may mean unreported data. These counts are checks, not automatic deletion rules.


In [ ]:
display(pd.DataFrame({
    "missing": movies[["budget", "revenue", "runtime", "release_date", "vote_average"]].isna().sum(),
    "zeros": movies[["budget", "revenue", "runtime", "release_date", "vote_average"]].eq(0).sum(),
}))
financial_mask = movies["budget"].gt(0) & movies["revenue"].gt(0)
print(f"Movies with positive budget and revenue: {financial_mask.sum():,}")


## Budget and revenue groups

Each unique movie belongs to one group below. The positive/positive group is the starting sample for calculating revenue divided by budget, subject to further validation. Zero values may indicate unreported data. No movies are removed.


In [ ]:
budget_positive = movies["budget"].gt(0)
revenue_positive = movies["revenue"].gt(0)
budget_zero = movies["budget"].eq(0)
revenue_zero = movies["revenue"].eq(0)

financial_groups = pd.DataFrame([
    ["Positive", "Positive", (budget_positive & revenue_positive).sum()],
    ["Zero", "Positive", (budget_zero & revenue_positive).sum()],
    ["Positive", "Zero", (budget_positive & revenue_zero).sum()],
    ["Zero", "Zero", (budget_zero & revenue_zero).sum()],
], columns=["Budget", "Revenue", "Movies"])

# Keep missing or negative values separate if they appear in a future dataset.
other_count = len(movies) - financial_groups["Movies"].sum()
if other_count:
    financial_groups.loc[len(financial_groups)] = ["Missing / invalid", "Either field", other_count]
financial_groups["Percent of movies"] = financial_groups["Movies"].div(len(movies)).mul(100)
financial_mask = budget_positive & revenue_positive
financial_groups.loc[len(financial_groups)] = ["TOTAL", "All groups", len(movies), 100.0]

assert financial_groups.iloc[:-1]["Movies"].sum() == len(movies)
display(financial_groups.style.hide(axis="index").format({
    "Movies": "{:,.0f}", "Percent of movies": "{:.1f}%",
}))


Budget,Revenue,Movies,Percent of movies
Positive,Positive,"6,841",53.5%
Zero,Positive,"2,027",15.9%
Positive,Zero,782,6.1%
Zero,Zero,"3,127",24.5%
TOTAL,All groups,"12,777",100.0%


## Next steps

Use `movies` for movie-level analysis and use `combined` for director–movie relationships. Director-history features and model training are not implemented yet. Financial eligibility and bust/average/hit boundaries still need to be finalized. The source CSVs remain unchanged.
